# Helsinki (FMI)

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load raw data

In [ ]:
import os
import pandas as pd
import numpy as np

DRIVE_ROOT = str(RAW_DIR)
HELSINKI_PATH = None

for root, dirs, files in os.walk(DRIVE_ROOT):
    for f in files:
        if 'Helsinki' in f and f.endswith('.csv'):
            HELSINKI_PATH = os.path.join(root, f)
            break
    if HELSINKI_PATH:
        break

if HELSINKI_PATH is None:
    raise FileNotFoundError("Helsinki CSV not found under data/raw.")

df_helsinki = pd.read_csv(
    HELSINKI_PATH,
    sep=';',
    header=67,
    parse_dates=['utctime'],
    index_col=['utctime'],
    dtype={'Remarks': str},
    low_memory=False
)

In [ ]:
SELECTED_COLS = {
    'pv_inv_out': 'Power_AC_W',
    'GLOBA_PT1M_AVG(:31)': 'Irradiance_POA',
    'GLOB_PT1M_AVG': 'GHI',
    'TTECH_PT1M_AVG(:32)': 'Temp_Module',
    'TA_PT1M_AVG(:31)': 'Temp_Ambient',
    'WS_PT10M_AVG': 'Wind_Speed',
    'RH_PT1M_AVG': 'Humidity',
    'vis_SnoP': 'Snow_Indicator'
}

df_sub = df_helsinki[list(SELECTED_COLS.keys())].rename(columns=SELECTED_COLS)

df_sub['Irradiance_POA'] = df_sub['Irradiance_POA'].clip(lower=0)
df_sub['GHI'] = df_sub['GHI'].clip(lower=0)
df_sub['Power_AC_W'] = df_sub['Power_AC_W'].clip(lower=0)

## Hourly aggregation (2019)

In [ ]:
df_helsinki_1h = df_sub.resample('1h').mean()

df_year = df_helsinki_1h.loc['2019-01-01 00:00:00':'2019-12-31 23:00:00'].copy()

for col in ['Power_AC_W', 'Irradiance_POA', 'GHI']:
    df_year[col] = df_year[col].fillna(0).clip(lower=0)

df_year = df_year.interpolate(method='time').bfill().ffill()

## Normalisation and features

In [ ]:
P_NOMINAL_AC = 20000.0
df_year['Power_PU'] = np.clip(df_year['Power_AC_W'] / P_NOMINAL_AC, 0.0, 1.0)

df_year['HOUR_SIN'] = np.sin(2 * np.pi * df_year.index.hour / 24.0)
df_year['HOUR_COS'] = np.cos(2 * np.pi * df_year.index.hour / 24.0)
df_year['MONTH_SIN'] = np.sin(2 * np.pi * df_year.index.month / 12.0)
df_year['MONTH_COS'] = np.cos(2 * np.pi * df_year.index.month / 12.0)

df_year['Latitude'] = 60.204
df_year['Longitude'] = 24.961
df_year['Tilt'] = 15.0

df_helsinki_final = df_year.drop(columns=['Temp_Module', 'Snow_Indicator'], errors='ignore')

df_helsinki_final = df_helsinki_final.rename(columns={'Power_AC_W': 'Power_Actual_W'})

FINAL_COLUMNS = [
    'Power_Actual_W', 'Power_PU',
    'Irradiance_POA', 'GHI', 'Temp_Ambient', 'Wind_Speed', 'Humidity',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
    'Latitude', 'Longitude', 'Tilt'
]
df_helsinki_final = df_helsinki_final[FINAL_COLUMNS]

## Save

In [ ]:
df_helsinki_final = df_helsinki_final.reset_index().rename(columns={'index': 'utctime'})

SAVE_PATH = OUT_DIR / 'Helsinki.csv'
df_helsinki_final.to_csv(SAVE_PATH, index=False)